In [3]:
import os
import sys
from pathlib import Path
import pprint as pp
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
DATA_FILE = Path(os.getcwd()) / "rag_data.txt"
TOP_K = 3
SIMILARITY_THRESHOLD = 0.3
LLM_API_KEY_ENV = "ZHIPU_API_KEY"   # ← 环境变量名，不是 Key 本身！
LLM_MODEL = "glm-4-flash"
FAISS_PATH = "./faiss_zh_index"                 # 本地向量库目录
EMBED_MODEL = "BAAI/bge-small-zh-v1.5"          # 编码模型
#EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DEVICE = "cuda"   # 没 GPU改 "cpu"

In [5]:
key = os.getenv(LLM_API_KEY_ENV)
if not key:
    print(f"❌ 环境变量 {LLM_API_KEY_ENV} 未设置")
    print(f"   请先设置你的智谱 API Key：")
    print(f"   Windows CMD:  set {LLM_API_KEY_ENV}=你的key")
    print(f"   PowerShell:   $env:{LLM_API_KEY_ENV}='你的key'")
    print(f"   获取地址: https://open.bigmodel.cn/")
    sys.exit(1)
else:
    pp.pprint(key)

'5e4e165df1c24405873fcb3621b5d7a0.6CEDD1OabBprR4p3'


In [14]:
# 读取文件
with open(DATA_FILE, "r", encoding="utf-8") as f:
    content = f.read()

In [7]:
# 按【文档N】切分
raw_chunks = content.split("【文档") # 以【文档 拆分字符串
documents = []
for chunk in raw_chunks:
    chunk = chunk.strip() #去除首尾空白字符
    if not chunk:
        continue
    text = "【文档" + chunk
    first_line = text.split("\n")[0]
    doc = Document( #     将一段文本封装成 LangChain 的 Document 对象，并用首行前30字符作为来源标识，便于后续检索结果溯源展示。
        page_content=text, # 指定该文档的正文文本内容
        metadata={"source": first_line} # 附加元数据：用首行标记来源
    )
    documents.append(doc)

print(f"   切分完成，共 {len(documents)} 个文档块")

   切分完成，共 8 个文档块


In [53]:
# 细分文档，每句规格不超过设定值
'''
把一整批 Document（比如好几篇长文）按“先按段落 → 再按句子 → 最后按字”的优先级切成小块，
每块最多 ~300 字符，相邻块之间重叠 50 字符，避免一句话被砍断导致检索丢上下文。
'''
text_splitter = RecursiveCharacterTextSplitter( # 递归字符分割器
    separators=["\n\n", "\n", "。", "！", "？",
                "；", "，", "、", " ", ""], # 切割优先级：先试双换行(段落) → 单换行 → 句号/感叹号/问号(句尾) → 逗号顿号 → 空格 → 最后实在不行就硬切字符
    chunk_size=20, #每个块的目标最大长度。按 length_function 算。如果能整段保留且不超 10，就整段保留，只有超了才递归切
    chunk_overlap=5,# 相邻 chunk 重叠5个字符，防止一句话被从中间劈开、检索时丢半句
    length_function=len #用来计算"这段文本有多长"的函数。len是内置函数，也可以自定义函数计算。若想按 token 可用 tokenizer
)
splits = text_splitter.split_documents(documents) # # 对 documents 批量切块。输入是 List[Document]，输出是 List[Document]（metadata 会原样带过去）
print(f"   文本已细分为 {len(splits)} 块（chunk_size=10, overlap=5）")


   文本已细分为 129 块（chunk_size=10, overlap=5）


In [31]:
"""
判断 FAISS 库是否已落盘。
LangChain 的 save_local 会写：index.faiss + index.pkl
两个都在才算“真存在”（只一个一定是上次写崩了）
"""
faiss_file = os.path.join(FAISS_PATH, "index.faiss")
pkl_file = os.path.join(FAISS_PATH, "index.pkl")
isExist = os.path.isdir(FAISS_PATH) and os.path.isfile(faiss_file) and os.path.isfile(pkl_file)
pp.pprint(isExist)

False


In [16]:
print(f"   加载嵌入模型: {EMBED_MODEL}")
'''
    它是 LangChain 提供的一个Embedding 模型包装类。作用是：把一段文本变成一串固定长度的数字向量（embedding），供后续 FAISS 做相似度检索。
    底层实际调用的是 sentence-transformers库
'''
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL, #指定用哪个预训练模型来把文本变成向量
    model_kwargs={"device": DEVICE}, #使用哪个设备计算向量
    encode_kwargs={"normalize_embeddings": True} #是否归一化
)

   加载嵌入模型: BAAI/bge-small-zh-v1.5


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

In [54]:
if isExist:
    vector_store = FAISS.load_local(
        FAISS_PATH,
        embeddings,
        allow_dangerous_deserialization=True # 是否允许反序列化操作
    )
else:
    # 创建向量数据库
    vector_store = FAISS.from_documents(splits, embeddings) #默认按照L2欧氏距离计算
    print("FAISS 索引类型:", type(vector_store.index).__name__)
    print("✅ 向量库构建完成")
    vector_store.save_local(FAISS_PATH)
    print(f"💾 已保存 FAISS 库到：{FAISS_PATH}（共 {vector_store.index.ntotal} 条向量）")

FAISS 索引类型: IndexFlatL2
✅ 向量库构建完成
💾 已保存 FAISS 库到：./faiss_zh_index（共 129 条向量）


In [39]:
# ========== 5. 检索 ==========
def retrieve(vector_store, query: str, top_k: int = TOP_K):
    docs_with_scores = vector_store.similarity_search_with_score(query, k=top_k) #去索引里找最像query的 k 个 → 返回 (Document, score) 元组

    print(f"\n{'='*60}")
    print(f"🔍 检索结果（Top {top_k}）：")
    print(f"{'='*60}")

    retrieved = []
    for i, (doc, score) in enumerate(docs_with_scores, 1):
        similarity = 1 / (1 + score)
        marker = "✅ 采纳" if similarity >= SIMILARITY_THRESHOLD else "⚠️ 低于阈值"
        print(f"   [{i}] {marker}  相似度≈{similarity:.3f}")
        print(f"       来源: {doc.metadata['source']}")
        print(f"       内容: {doc.page_content}")
        if similarity >= SIMILARITY_THRESHOLD:
            retrieved.append(doc)

    if not retrieved:
        print(f"\n   ❌ 没有找到超过阈值({SIMILARITY_THRESHOLD})的片段")
    return retrieved


In [33]:
# ========== 6. 拼接 Prompt ==========
def build_prompt(query: str, retrieved_docs) -> str:
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        context_parts.append(f"--- 资料{i}（{doc.metadata['source']}）---\n{doc.page_content}")
    context = "\n\n".join(context_parts)

    prompt = f"""【系统指令】
你是一个严谨的问答助手。请严格基于下面提供的资料回答问题。

规则：
1. 只使用资料中的信息回答，不要编造或补充资料外的内容
2. 如果资料中没有明确答案，必须回答"不知道"
3. 回答要简洁，直接给结论，必要时引用资料原文
4. 如果多份资料有矛盾，指出矛盾并说明各资料内容
5. 回答前，先列出资料中的相关条件，再逐一比对用户情况，最后给出结论

【参考资料】
{context}

【问题】
{query}

【回答】"""

    print(f"\n{'='*60}")
    print(f"📝 准备传给 LLM 的完整 Prompt（{len(prompt)} 字符）：")
    print(f"{'='*60}")
    print(prompt)
    print(f"{'='*60}")
    return prompt

In [46]:
# ========== 8. 模式2：RAG + LLM ==========
def mode_rag_with_llm(query: str, retrieved_docs) -> str:
    print(f"\n{'='*60}")
    print(f"🤖 模式2：RAG + LLM（智谱 {LLM_MODEL}）")
    print(f"{'='*60}")

    if not retrieved_docs:
        msg = "不知道。资料库中没有找到与您问题相关的信息。"
        print(f"\n   → {msg}")
        return msg

    print(f"\n   📄 检索到的原文片段：")
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\n   ┌─ 片段{i}（{doc.metadata['source']}）")
        for line in doc.page_content.split("\n"):
            print(f"   │  {line}")
        print(f"   └─")

    prompt = build_prompt(query, retrieved_docs)

    print(f"\n   ⏳ 正在调用 {LLM_MODEL} 生成回答...")
    try:
        from zai import ZhipuAiClient
    except ImportError:
        print("❌ 缺少依赖，请安装: pip install zhipuai")
        sys.exit(1)

    # ✅ 从环境变量读 Key
    api_key = os.getenv(LLM_API_KEY_ENV)
    print(f"   🔑 API Key 前5位: {api_key[:5]}..." if api_key else "   🔑 API Key 为空！！")
    client = ZhipuAiClient(api_key=api_key)

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "你是一个严谨的问答助手。严格基于用户提供的资料回答问题，不编造。资料中没有的答案就说'不知道'。"},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=1024
        )
        answer = response.choices[0].message.content
    except Exception as e:
        answer = f"调用 LLM 出错: {e}"
        print(f"   ❌ {answer}")

    print(f"\n   ✅ LLM 返回回答:")
    print(f"   {'─'*50}")
    for line in answer.split("\n"):
        print(f"   {line}")
    print(f"   {'─'*50}")



In [55]:
# 使用 BAAI/bge-small-zh-v1.5 编码模型时，查询问题时最好拼上下面这句话，因为模型训练时带着这句话训练的，加上效果会好一点
# query = "你是谁"  → 实际检索时写成 f"Represent this sentence for searching relevant passages: {query}"
# 存文档不用加；只查的时候加
query = "我入职三年，可以休几天年假？"
docs = retrieve(vector_store, query)


🔍 检索结果（Top 3）：
   [1] ✅ 采纳  相似度≈0.630
       来源: 【文档1：公司年假制度】
       内容: 员工入职满1年可享受5天年假
   [2] ✅ 采纳  相似度≈0.575
       来源: 【文档1：公司年假制度】
       内容: ，满3年享受10天，满5年享受15天
   [3] ✅ 采纳  相似度≈0.566
       来源: 【文档1：公司年假制度】
       内容: 年假需在当年12月31日前使用完毕


In [56]:
mode_rag_with_llm(query, docs)


🤖 模式2：RAG + LLM（智谱 glm-4-flash）

   📄 检索到的原文片段：

   ┌─ 片段1（【文档1：公司年假制度】）
   │  员工入职满1年可享受5天年假
   └─

   ┌─ 片段2（【文档1：公司年假制度】）
   │  ，满3年享受10天，满5年享受15天
   └─

   ┌─ 片段3（【文档1：公司年假制度】）
   │  年假需在当年12月31日前使用完毕
   └─

📝 准备传给 LLM 的完整 Prompt（346 字符）：
【系统指令】
你是一个严谨的问答助手。请严格基于下面提供的资料回答问题。

规则：
1. 只使用资料中的信息回答，不要编造或补充资料外的内容
2. 如果资料中没有明确答案，必须回答"不知道"
3. 回答要简洁，直接给结论，必要时引用资料原文
4. 如果多份资料有矛盾，指出矛盾并说明各资料内容
5. 回答前，先列出资料中的相关条件，再逐一比对用户情况，最后给出结论

【参考资料】
--- 资料1（【文档1：公司年假制度】）---
员工入职满1年可享受5天年假

--- 资料2（【文档1：公司年假制度】）---
，满3年享受10天，满5年享受15天

--- 资料3（【文档1：公司年假制度】）---
年假需在当年12月31日前使用完毕

【问题】
我入职三年，可以休几天年假？

【回答】

   ⏳ 正在调用 glm-4-flash 生成回答...
   🔑 API Key 前5位: 5e4e1...

   ✅ LLM 返回回答:
   ──────────────────────────────────────────────────
   根据资料1和资料2，入职满3年的员工可以享受10天年假。
   ──────────────────────────────────────────────────
